In [4]:
# ==========================================
# IDEAM SCRAPER + CHUNKER (NOTEBOOK VERSION)
# ==========================================

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import json
import os
import time
import pandas as pd

# =========================
# PATH SETUP (SAFE)
# =========================

PROJECT_ROOT = os.getcwd()  # assumes notebook is in project root
DATA_RAW = os.path.join(PROJECT_ROOT, "data", "raw")
DATA_PROCESSED = os.path.join(PROJECT_ROOT, "data", "processed")

os.makedirs(DATA_RAW, exist_ok=True)
os.makedirs(DATA_PROCESSED, exist_ok=True)

RAW_FILE = os.path.join(DATA_RAW, "ideam_raw.json")
CHUNK_FILE = os.path.join(DATA_PROCESSED, "ideam_chunks.csv")

print("Saving raw to:", RAW_FILE)
print("Saving chunks to:", CHUNK_FILE)

# =========================
# SCRAPER
# =========================

START_URL = "https://ideam.ie"
MAX_PAGES = 40

visited = set()
structured_data = []

def clean_text(text):
    return " ".join(text.split()).strip()

def extract_structured_content(url):
    try:
        res = requests.get(url, timeout=10)
        if res.status_code != 200:
            return None

        soup = BeautifulSoup(res.text, "html.parser")

        for tag in soup(["nav", "footer", "header", "script", "style", "form"]):
            tag.decompose()

        page_title = soup.title.string.strip() if soup.title else ""

        page_content = []
        current_heading = ""

        for element in soup.find_all(["h1", "h2", "h3", "p", "li"]):
            text = clean_text(element.get_text())

            if len(text) < 30:
                continue

            if element.name in ["h1", "h2", "h3"]:
                current_heading = text
            else:
                page_content.append({
                    "heading": current_heading,
                    "text": text
                })

        return {
            "url": url,
            "title": page_title,
            "content": page_content
        }

    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return None


def crawl_site(start_url):
    domain = urlparse(start_url).netloc
    to_visit = [start_url]

    while to_visit and len(visited) < MAX_PAGES:
        url = to_visit.pop(0)

        if url in visited:
            continue

        visited.add(url)
        print("Crawling:", url)

        page_data = extract_structured_content(url)
        if page_data:
            structured_data.append(page_data)

        try:
            res = requests.get(url, timeout=10)
            soup = BeautifulSoup(res.text, "html.parser")

            for a in soup.find_all("a", href=True):
                link = urljoin(url, a["href"])
                if domain in link and link not in visited and "#" not in link:
                    to_visit.append(link)
        except:
            pass

        time.sleep(0.3)

    print("\nTotal pages scraped:", len(structured_data))


# =========================
# CHUNKER
# =========================

CHUNK_SIZE = 500

def chunk_text_blocks(page):
    chunks = []
    buffer = ""
    word_count = 0

    for block in page["content"]:
        heading = block.get("heading", "")
        text = block.get("text", "")

        combined = f"{heading}. {text}" if heading else text
        words = combined.split()

        if word_count + len(words) <= CHUNK_SIZE:
            buffer += " " + combined
            word_count += len(words)
        else:
            chunks.append(buffer.strip())
            buffer = combined
            word_count = len(words)

    if buffer:
        chunks.append(buffer.strip())

    return chunks


# =========================
# EXECUTION
# =========================

crawl_site(START_URL)

# Save raw JSON
with open(RAW_FILE, "w", encoding="utf-8") as f:
    json.dump(structured_data, f, indent=2, ensure_ascii=False)

print("\nRaw data saved.")

# Create chunks
all_chunks = []

for page in structured_data:
    page_url = page["url"]
    title = page.get("title", "")

    page_chunks = chunk_text_blocks(page)

    for chunk in page_chunks:
        if len(chunk.split()) > 50:
            all_chunks.append({
                "page_url": page_url,
                "title": title,
                "content": chunk
            })

df = pd.DataFrame(all_chunks)
df.drop_duplicates(subset=["content"], inplace=True)
df.reset_index(drop=True, inplace=True)

print("After deduplication:", len(df))
df.to_csv(CHUNK_FILE, index=False)


print("\nChunks saved.")
print("Total chunks:", len(df))
print("Average words per chunk:", df["content"].apply(lambda x: len(x.split())).mean())

df.head()

Saving raw to: C:\Users\user\chatbot_postgresql_test\scrapers\data\raw\ideam_raw.json
Saving chunks to: C:\Users\user\chatbot_postgresql_test\scrapers\data\processed\ideam_chunks.csv
Crawling: https://ideam.ie
Crawling: https://ideam.ie/
Crawling: https://ideam.ie/ideam-institute/about-us/
Crawling: https://ideam.ie/ideam-institute/initiatives/research-projects/
Crawling: https://ideam.ie/ideam-institute/initiatives/learning-factory/
Crawling: https://ideam.ie/events/
Crawling: https://ideam.ie/ideam-institute/resources/publications-and-reports/
Crawling: https://ideam.ie/ideam-institute/ideam-cluster-media-gallery/
Crawling: https://ideam.ie/ideam-institute/contact-institute/
Crawling: https://ideam.ie/ideam-institute/initiatives/learning-factory/students-and-alumni/
Crawling: https://ideam.ie/ideam-cluster/
Crawling: https://ideam.ie/ideam-cluster/ideam-members/become-a-member/
Crawling: https://ideam.ie/ideam-cluster/ideam-members/
Crawling: https://ideam.ie/ideam-cluster/business-g

,page_url,title,content
0,https://ideam.ie,IDEAM Research Institute | IDEAM,"Tuition WaiverFee Waiver Valued at €5,500. Bac..."
1,https://ideam.ie,IDEAM Research Institute | IDEAM,IDEAM Research Institute Supports. IDEAM provi...
2,https://ideam.ie/ideam-institute/about-us/,About Us | IDEAM,"Welcome to IDEAM Institute, where innovation m..."
3,https://ideam.ie/ideam-institute/about-us/,About Us | IDEAM,"Welcome to IDEAM Institute, where innovation m..."
4,https://ideam.ie/ideam-institute/about-us/,About Us | IDEAM,"Welcome to IDEAM Institute, where innovation m..."


In [7]:
Project root added: C:\Users\user\chatbot_postgresql_test\scrapers

Project root added: C:\Users\user\chatbot_postgresql_test\scrapers


In [6]:
from app.database.ingest import ingest_chunks
ingest_chunks()

ModuleNotFoundError: No module named 'app'